# 05 — Stage 2: GATv2 Graph Branch + Post-hoc Calibration

**Project:** HierFuse — Hierarchical PowerShell / LotL Detection with Learned Fusion  
**Stage:** 2 of 3 — structural encoder (graph modality)  
**Platform:** Kaggle Free Tier · P100 GPU recommended  
**Runtime:** ~2–3 hours  
**Inputs:** `02-splits-dataset`, MAGIC pre-processed DARPA TC E3-CADETS pkl files  
**Outputs:** `/kaggle/working/gat/`

## Architecture

**GATv2** (Brody et al. 2022) over DARPA TC E3-CADETS provenance graphs  
pre-processed by MAGIC (USENIX Security 2024).

- 2× GATv2Conv layers, 4 heads, hidden_dim=128
- Global mean + max pooling → 256-d graph embedding
- Returns: probability + 256-d embedding → Stage 3 fusion

**Graph construction fallback:** If MAGIC pkl files are unavailable,  
a token co-occurrence proxy graph is constructed from raw PowerShell text  
(node features: type one-hot + SHA-256 hash embedding).

## Post-hoc calibration (merged from 05b — no separate notebook needed)

GATv2 trains on 50:50 data but evaluates at C2 test (≤10% malicious).  
At 10% prevalence, threshold=0.5 is miscalibrated → C2 F1 collapses.  
**Platt scaling** (logistic regression on val probs) is fit per-seed and saves  
calibrated prob arrays alongside raw ones. Stage 3 fusion uses calibrated probs.


## Cell 1 — Bootstrap: locate datasets, create output dirs

In [1]:
import os, sys, json, time, gc, warnings, random, hashlib
from pathlib import Path
from datetime import datetime
import subprocess

warnings.filterwarnings('ignore')
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run ID: {RUN_ID}')

def _rglob_first(root, pattern):
    for p in Path(root).rglob(pattern): return p
    return None

KI = Path('/kaggle/input')
SPLITS_DIR   = None
for m in KI.rglob('train_seed42.parquet'): SPLITS_DIR = m.parent; break
WEIGHTS_PATH = _rglob_first(KI, 'class_weights_per_seed.json')
MAGIC_PKL = _rglob_first(KI, 'ps_ast_graphs.pkl')    # DARPA TC E3-CADETS MAGIC pkl (optional)

print()
for label, path in [('Splits dir', SPLITS_DIR), ('Weights', WEIGHTS_PATH),
                     ('MAGIC pkl (optional)', MAGIC_PKL)]:
    ok = path is not None and Path(path).exists()
    print(f'  {"[OK]" if ok else "[MISSING]":<12} {label}: {path}')

if not SPLITS_DIR or not WEIGHTS_PATH:
    raise FileNotFoundError('Attach 02-splits-dataset before running.')

with open(WEIGHTS_PATH) as f: CLASS_WEIGHTS = json.load(f)

OUT_ROOT   = Path('/kaggle/working/gat')
MODELS_DIR = OUT_ROOT / 'checkpoints'
PROBS_DIR  = OUT_ROOT / 'probs'
EMBS_DIR   = OUT_ROOT / 'embeddings'
CAL_DIR    = OUT_ROOT / 'calibrators'
for d in [MODELS_DIR, PROBS_DIR, EMBS_DIR, CAL_DIR]: d.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 1337, 2024]
if MAGIC_PKL:
    print(f'  [INFO]  MAGIC pkl found: {MAGIC_PKL}  — using AST graphs')
else:
    print('  [INFO]  MAGIC pkl not found — will use text-proxy co-occurrence graphs')
print(f'Outputs: {OUT_ROOT}')
print('Cell 1 OK.')


Run ID: 20260603_210958

  [OK]         Splits dir: /kaggle/input/datasets/onkarkmane1501/02-splits-dataset/splits
  [OK]         Weights: /kaggle/input/datasets/onkarkmane1501/02-splits-dataset/splits/class_weights_per_seed.json
  [OK]         MAGIC pkl (optional): /kaggle/input/datasets/onkarkmane1501/04b-ast-extraction-dataset/ps_ast_graphs.pkl
  [INFO]  MAGIC pkl found: /kaggle/input/datasets/onkarkmane1501/04b-ast-extraction-dataset/ps_ast_graphs.pkl  — using AST graphs
Outputs: /kaggle/working/gat
Cell 1 OK.


## Cell 2 — Hyperparameters

In [2]:
GAT_HIDDEN_DIM      = 128
GAT_HEADS           = 4
GAT_NUM_LAYERS      = 2
GAT_DROPOUT         = 0.1
GAT_NEG_SLOPE       = 0.2
GRAPH_EMB_DIM       = 2 * GAT_HIDDEN_DIM     # 256
NODE_TYPE_VOCAB_CAP = 200
TOKEN_HASH_DIM      = 32
NODE_FEAT_DIM       = NODE_TYPE_VOCAB_CAP + TOKEN_HASH_DIM  # 232
MAX_NODES           = 800

LR           = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS       = 50
PATIENCE     = 10
BATCH_SIZE   = 32

GLOBAL_T0 = time.time()
print('Hyperparameters:')
for k,v in dict(HIDDEN=GAT_HIDDEN_DIM, HEADS=GAT_HEADS, LAYERS=GAT_NUM_LAYERS,
                EMB_DIM=GRAPH_EMB_DIM, NODE_FEAT_DIM=NODE_FEAT_DIM,
                DROPOUT=GAT_DROPOUT, LR=LR, EPOCHS=EPOCHS, PATIENCE=PATIENCE).items():
    print(f'  {k:<16}: {v}')


Hyperparameters:
  HIDDEN          : 128
  HEADS           : 4
  LAYERS          : 2
  EMB_DIM         : 256
  NODE_FEAT_DIM   : 232
  DROPOUT         : 0.1
  LR              : 0.001
  EPOCHS          : 50
  PATIENCE        : 10


## Cell 3 — Installs and imports

In [3]:
subprocess.run([sys.executable,'-m','pip','install','-q',
                '--disable-pip-version-check',
                'torch>=2.0','torch-geometric','scikit-learn','pyarrow'],
               check=True, capture_output=True)


import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, pickle
from torch_geometric.nn  import GATv2Conv, global_mean_pool, global_max_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as PyGDataLoader
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                              brier_score_loss, roc_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('gat')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
log.info(f'Device: {DEVICE}')
print('Cell 3 OK.')


21:10:30 | INFO | Device: cuda


Cell 3 OK.


## Cell 4 — Graph construction (MAGIC pkl or text-proxy fallback)

In [4]:
import re as _re

_PS_TOKEN_RE = _re.compile(r'[A-Za-z_\-][\w\-]*|"[^"]*"|\'[^\']*\'|\$\w+|[\(\)\{\}\[\]\|;]')

def _hash_token_to_vec(token, dim=TOKEN_HASH_DIM):
    """sha256 + tanh: exact match to NB05 original."""
    if not token: return np.zeros(dim, dtype=np.float32)
    h = hashlib.sha256(token.encode('utf-8', errors='replace')).digest()
    needed = dim * 4
    if len(h) < needed: h = (h * (needed//len(h)+1))[:needed]
    arr = np.frombuffer(h[:needed], dtype=np.int32).astype(np.float32)
    return np.tanh(arr / (2**24))

def text_to_graph(script_text, label=None, max_nodes=MAX_NODES):
    """Token co-occurrence proxy graph (fallback when MAGIC pkl unavailable)."""
    tokens = _PS_TOKEN_RE.findall(script_text[:8192])
    if not tokens: tokens = ['<empty>']
    tokens = tokens[:max_nodes]; n = len(tokens)
    type_ids = np.array([int(hashlib.md5(t.lower().encode()).hexdigest(),16) % NODE_TYPE_VOCAB_CAP
                         for t in tokens], dtype=np.int32)
    hash_cache = np.zeros((n, TOKEN_HASH_DIM), dtype=np.float16)
    for i, t in enumerate(tokens): hash_cache[i] = _hash_token_to_vec(t).astype(np.float16)
    x = np.zeros((n, NODE_FEAT_DIM), dtype=np.float16)
    x[np.arange(n), type_ids] = 1.0
    x[:, NODE_TYPE_VOCAB_CAP:] = hash_cache
    if n > 1:
        src = np.arange(n-1, dtype=np.int64); dst = np.arange(1, n, dtype=np.int64)
        edges = np.stack([np.concatenate([src,dst]), np.concatenate([dst,src])], axis=0)
    else:
        edges = np.array([[0],[0]], dtype=np.int64)
    g = Data(x=torch.from_numpy(x.astype(np.float32)), edge_index=torch.from_numpy(edges))
    if label is not None: g.y = torch.tensor([int(label)], dtype=torch.long)
    return g

def load_graphs_for_split(df, label_col='label', magic_pkl_path=None):
    """Returns list of PyG Data objects."""
    if magic_pkl_path is not None and Path(magic_pkl_path).exists():
        try:
            with open(magic_pkl_path,'rb') as f: raw = pickle.load(f)
            if isinstance(raw, dict):
                graphs = []
                for _, row in df.iterrows():
                    g = raw.get(row.get('sha256',''), text_to_graph(str(row.get('script_text',''))))
                    g.y = torch.tensor([int(row[label_col])], dtype=torch.long)
                    graphs.append(g)
                return graphs
        except Exception as e:
            log.warning(f'MAGIC pkl load failed ({e}) — falling back to text-proxy graphs')
    graphs = []
    for _, row in df.iterrows():
        g = text_to_graph(str(row.get('script_text','')), label=int(row[label_col]))
        graphs.append(g)
    return graphs

print('Graph construction helpers defined.')


Graph construction helpers defined.


## Cell 5 — GATv2 classifier

In [5]:
class GATv2Classifier(nn.Module):
    """GATv2 with residual + global mean/max pooling → 256-d graph embedding."""
    def __init__(self, in_dim=NODE_FEAT_DIM, hidden=GAT_HIDDEN_DIM, heads=GAT_HEADS,
                 num_layers=GAT_NUM_LAYERS, dropout=GAT_DROPOUT, neg_slope=GAT_NEG_SLOPE):
        super().__init__()
        self.dropout_p = dropout
        self.gat1      = GATv2Conv(in_dim,        hidden, heads=heads, concat=True,
                                    dropout=dropout, negative_slope=neg_slope, add_self_loops=True)
        self.gat2      = GATv2Conv(hidden*heads,  hidden, heads=heads, concat=False,
                                    dropout=dropout, negative_slope=neg_slope, add_self_loops=True)
        self.skip_proj = nn.Linear(hidden*heads, hidden)
        self.head_drop = nn.Dropout(dropout)
        self.head      = nn.Linear(GRAPH_EMB_DIM, 1)

    def encode(self, x, edge_index, batch):
        x = x.float()
        h1 = F.elu(self.gat1(x, edge_index))
        h1 = F.dropout(h1, p=self.dropout_p, training=self.training)
        h2 = self.gat2(h1, edge_index) + self.skip_proj(h1)
        h  = F.elu(h2)
        return torch.cat([global_mean_pool(h,batch), global_max_pool(h,batch)], dim=-1)

    def forward(self, x, edge_index, batch, labels=None, pos_weight=None):
        emb    = self.encode(x, edge_index, batch)
        logits = self.head(self.head_drop(emb)).squeeze(-1)
        loss   = None
        if labels is not None:
            pw  = torch.tensor([pos_weight], device=logits.device) if pos_weight else None
            loss= (nn.BCEWithLogitsLoss(pos_weight=pw) if pw else nn.BCEWithLogitsLoss())(logits, labels.float())
        return {'logits': logits, 'embedding': emb, 'loss': loss}


def gat_eval(model, loader, pos_weight=None):
    """Returns (metrics_dict, probs_np, embs_np)."""
    model.eval()
    all_probs=[]; all_labels=[]; all_embs=[]
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            out   = model(batch.x.float(), batch.edge_index, batch.batch)
            probs = torch.sigmoid(out['logits'].float()).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(batch.y.cpu().numpy().tolist())
            all_embs.append(out['embedding'].cpu().numpy())
    y=np.array(all_labels); yp=np.array(all_probs)
    yb=(yp>=0.5).astype(int)
    fpr_a, tpr_a, _ = roc_curve(y, yp)
    metrics = {
        'f1':          round(float(f1_score(y,yb,zero_division=0)),4),
        'auroc':       round(float(roc_auc_score(y,yp)),4),
        'pr_auc':      round(float(average_precision_score(y,yp)),4),
        'tpr_at_1fpr': round(float(np.interp(0.01,fpr_a,tpr_a)),4),
    }
    return metrics, yp, np.vstack(all_embs)

print('GATv2Classifier and eval helper defined.')


GATv2Classifier and eval helper defined.


## Cell 6 — Training loop (3 seeds) + post-hoc Platt calibration

In [6]:
all_gat_results = {}

for seed in SEEDS:
    log.info(f"\n{'='*60}\nSEED {seed}\n{'='*60}")
    t0 = time.time()
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    w   = CLASS_WEIGHTS[str(seed)]
    spw = float(w.get('scale_pos_weight', 1.0))

    # Build graphs
    log.info('  Building graphs...')
    df_tr   = pd.read_parquet(SPLITS_DIR/f'train_seed{seed}.parquet')
    df_va   = pd.read_parquet(SPLITS_DIR/f'val_seed{seed}.parquet')
    df_te   = pd.read_parquet(SPLITS_DIR/f'test_seed{seed}.parquet')
    df_tec2 = pd.read_parquet(SPLITS_DIR/f'test_c2_seed{seed}.parquet')

    g_tr   = load_graphs_for_split(df_tr,   magic_pkl_path=MAGIC_PKL)
    g_va   = load_graphs_for_split(df_va,   magic_pkl_path=MAGIC_PKL)
    g_te   = load_graphs_for_split(df_te,   magic_pkl_path=MAGIC_PKL)
    g_tec2 = load_graphs_for_split(df_tec2, magic_pkl_path=MAGIC_PKL)
    in_channels = g_tr[0].x.shape[1]
    log.info(f'  Graphs built: in_channels={in_channels}')

    tr_ld   = PyGDataLoader(g_tr,   batch_size=BATCH_SIZE, shuffle=True)
    va_ld   = PyGDataLoader(g_va,   batch_size=BATCH_SIZE, shuffle=False)
    te_ld   = PyGDataLoader(g_te,   batch_size=BATCH_SIZE, shuffle=False)
    tec2_ld = PyGDataLoader(g_tec2, batch_size=BATCH_SIZE, shuffle=False)

    model     = GATv2Classifier(in_dim=in_channels).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([spw], device=DEVICE))

    best_val_auroc = 0.0
    best_ckpt      = MODELS_DIR / f'gat_seed{seed}_best.pt'
    patience_cnt   = 0

    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss = 0.0
        for batch in tr_ld:
            batch = batch.to(DEVICE); optimizer.zero_grad()
            out   = model(batch.x.float(), batch.edge_index, batch.batch,
                          labels=batch.y, pos_weight=spw)
            out['loss'].backward(); optimizer.step()
            total_loss += out['loss'].item()

        va_metrics, _, _ = gat_eval(model, va_ld)
        if epoch % 5 == 0 or epoch == 1:
            log.info(f'  Epoch {epoch:>3}/{EPOCHS}  loss={total_loss/len(tr_ld):.4f}  '
                     f'val_F1={va_metrics["f1"]:.4f}  val_AUROC={va_metrics["auroc"]:.4f}')
        if va_metrics['auroc'] > best_val_auroc:
            best_val_auroc = va_metrics['auroc']
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'val_auroc': best_val_auroc}, best_ckpt)
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE: log.info(f'  Early stopping at epoch {epoch}'); break

    # Load best checkpoint
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])

    tr_m,   tr_probs,   tr_embs   = gat_eval(model, tr_ld)
    va_m,   va_probs,   va_embs   = gat_eval(model, va_ld)
    te_m,   te_probs,   te_embs   = gat_eval(model, te_ld)
    tec2_m, tec2_probs, tec2_embs = gat_eval(model, tec2_ld)
    log.info(f'  [balanced test] F1={te_m["f1"]:.4f}  AUROC={te_m["auroc"]:.4f}')
    log.info(f'  [C2 raw]        F1={tec2_m["f1"]:.4f}  AUROC={tec2_m["auroc"]:.4f}')

    # ── Post-hoc Platt calibration ────────────────────────────────────────
    val_labels = df_va['label'].values
    platt = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    platt.fit(va_probs.reshape(-1,1), val_labels)
    iso   = IsotonicRegression(out_of_bounds='clip')
    iso.fit(va_probs, val_labels)
    platt_brier = brier_score_loss(val_labels, platt.predict_proba(va_probs.reshape(-1,1))[:,1])
    iso_brier   = brier_score_loss(val_labels, iso.predict(va_probs))
    best_cal    = 'platt' if platt_brier <= iso_brier else 'isotonic'
    log.info(f'  Calibrator: {best_cal} (Platt Brier={platt_brier:.4f}, Iso={iso_brier:.4f})')

    def apply_cal(raw):
        if best_cal == 'platt':
            return platt.predict_proba(raw.reshape(-1,1))[:,1].astype(np.float32)
        return iso.predict(raw).astype(np.float32)

    calibrators = {'platt':platt,'isotonic':iso,'best':best_cal,
                   'brier_platt':platt_brier,'brier_iso':iso_brier}
    with open(CAL_DIR/f'calibrators_seed{seed}.pkl','wb') as f:
        pickle.dump(calibrators, f, protocol=4)

    # Calibrated C2 metrics
    tec2_cal = apply_cal(tec2_probs)
    cal_f1   = round(float(f1_score(df_tec2['label'].values,(tec2_cal>=0.5).astype(int),zero_division=0)),4)
    cal_auroc= round(float(roc_auc_score(df_tec2['label'].values, tec2_cal)),4)
    log.info(f'  [C2 calibrated] F1={cal_f1:.4f}  AUROC={cal_auroc:.4f}')

    # Save raw + calibrated probs, embeddings
    for split_name, raw, embs in [('train',tr_probs,tr_embs),('val',va_probs,va_embs),
                                   ('test',te_probs,te_embs),('test_c2',tec2_probs,tec2_embs)]:
        np.save(PROBS_DIR/f'{split_name}_probs_seed{seed}.npy',     raw)
        np.save(PROBS_DIR/f'{split_name}_probs_cal_seed{seed}.npy', apply_cal(raw))
        np.save(EMBS_DIR/f'{split_name}_emb_seed{seed}.npy',         embs)

    all_gat_results[str(seed)] = {
        'train_metrics':      tr_m, 'val_metrics': va_m,
        'test_metrics':       te_m, 'test_c2_metrics': tec2_m,
        'test_c2_calibrated': {'f1':cal_f1,'auroc':cal_auroc},
        'calibrator_best': best_cal, 'brier_platt': platt_brier, 'brier_iso': iso_brier,
        'elapsed_min': round((time.time()-t0)/60,1),
    }
    del model, g_tr, g_va, g_te, g_tec2
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

results_path = OUT_ROOT / 'gat_results.json'
with open(results_path,'w') as f: json.dump(all_gat_results, f, indent=2)
log.info(f'Results saved: {results_path}')
log.info(f'Total: {(time.time()-GLOBAL_T0)/60:.1f} min')
print('Cell 6 OK.')


21:10:30 | INFO | 
SEED 42
21:10:30 | INFO |   Building graphs...
21:12:59 | INFO |   Graphs built: in_channels=232
21:13:22 | INFO |   Epoch   1/50  loss=0.4491  val_F1=0.8230  val_AUROC=0.9023
21:14:49 | INFO |   Epoch   5/50  loss=0.3502  val_F1=0.8405  val_AUROC=0.9243
21:16:40 | INFO |   Epoch  10/50  loss=0.3320  val_F1=0.8586  val_AUROC=0.9326
21:18:30 | INFO |   Epoch  15/50  loss=0.3204  val_F1=0.8168  val_AUROC=0.9330
21:20:22 | INFO |   Epoch  20/50  loss=0.3107  val_F1=0.8540  val_AUROC=0.9333
21:22:12 | INFO |   Epoch  25/50  loss=0.3059  val_F1=0.8581  val_AUROC=0.9384
21:24:03 | INFO |   Epoch  30/50  loss=0.3002  val_F1=0.8613  val_AUROC=0.9380
21:25:54 | INFO |   Epoch  35/50  loss=0.2998  val_F1=0.8425  val_AUROC=0.9346
21:27:46 | INFO |   Epoch  40/50  loss=0.3015  val_F1=0.8633  val_AUROC=0.9373
21:28:30 | INFO |   Early stopping at epoch 42
21:28:45 | INFO |   [balanced test] F1=0.8682  AUROC=0.9423
21:28:45 | INFO |   [C2 raw]        F1=0.6287  AUROC=0.9327
21:28:

Cell 6 OK.


## Cell 7 — Calibration summary + results table + sanity checks

In [7]:
print('\n=== GAT Branch Results (mean ± std across 3 seeds) ===')
for vkey, vlabel in [('test_metrics','Balanced test'),
                     ('test_c2_metrics','C2 raw'),('test_c2_calibrated','C2 calibrated')]:
    print(f'\n[{vlabel}]')
    for metric in ['f1','auroc']:
        vals = [all_gat_results[str(s)][vkey].get(metric,0) for s in SEEDS if str(s) in all_gat_results]
        if vals: print(f'  {metric:<10}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

print('\n=== Calibration Delta (C2 F1) ===')
for seed in SEEDS:
    if str(seed) not in all_gat_results: continue
    raw = all_gat_results[str(seed)]['test_c2_metrics']['f1']
    cal = all_gat_results[str(seed)]['test_c2_calibrated']['f1']
    print(f'  Seed {seed}: raw={raw:.4f}  calibrated={cal:.4f}  delta={cal-raw:+.4f}')

# Sanity checks
ck_ok=ck_fail=0
def chk(c,n,d=''):
    global ck_ok,ck_fail
    if c: ck_ok+=1; print(f'[OK]   {n}')
    else: ck_fail+=1; print(f'[FAIL] {n}  {d}')

print('\n=== SANITY CHECKS ===')
chk((OUT_ROOT/'gat_results.json').exists(), 'gat_results.json saved')
for seed in SEEDS:
    if str(seed) not in all_gat_results: continue
    for split in ['train','val','test','test_c2']:
        chk((PROBS_DIR/f'{split}_probs_seed{seed}.npy').exists(),     f'Seed {seed} {split} raw probs')
        chk((PROBS_DIR/f'{split}_probs_cal_seed{seed}.npy').exists(), f'Seed {seed} {split} cal probs')
        emb = EMBS_DIR/f'{split}_emb_seed{seed}.npy'
        if emb.exists():
            chk(np.load(emb).shape[1]==256, f'Seed {seed} {split} emb dim=256')
    chk((CAL_DIR/f'calibrators_seed{seed}.pkl').exists(), f'Seed {seed} calibrator saved')
    m = all_gat_results[str(seed)]['test_metrics']
    chk(m['auroc']>0.90, f'Seed {seed} balanced test AUROC>0.90', f'{m["auroc"]:.4f}')

print(f'\n{ck_ok} passed, {ck_fail} failed')
if ck_fail == 0:
    print('\nALL CHECKS PASSED.')
    print('Next: upload /kaggle/working/gat/ as Kaggle Dataset "05-gat-dataset"')
    print('Then run 06_fusion.ipynb')
else:
    raise RuntimeError(f'{ck_fail} checks FAILED.')



=== GAT Branch Results (mean ± std across 3 seeds) ===

[Balanced test]
  f1        : 0.8680 ± 0.0018
  auroc     : 0.9456 ± 0.0026

[C2 raw]
  f1        : 0.6493 ± 0.0534
  auroc     : 0.9462 ± 0.0104

[C2 calibrated]
  f1        : 0.6120 ± 0.0486
  auroc     : 0.9447 ± 0.0098

=== Calibration Delta (C2 F1) ===
  Seed 42: raw=0.6287  calibrated=0.6138  delta=-0.0149
  Seed 1337: raw=0.5966  calibrated=0.6706  delta=+0.0740
  Seed 2024: raw=0.7225  calibrated=0.5516  delta=-0.1709

=== SANITY CHECKS ===
[OK]   gat_results.json saved
[OK]   Seed 42 train raw probs
[OK]   Seed 42 train cal probs
[OK]   Seed 42 train emb dim=256
[OK]   Seed 42 val raw probs
[OK]   Seed 42 val cal probs
[OK]   Seed 42 val emb dim=256
[OK]   Seed 42 test raw probs
[OK]   Seed 42 test cal probs
[OK]   Seed 42 test emb dim=256
[OK]   Seed 42 test_c2 raw probs
[OK]   Seed 42 test_c2 cal probs
[OK]   Seed 42 test_c2 emb dim=256
[OK]   Seed 42 calibrator saved
[OK]   Seed 42 balanced test AUROC>0.90
[OK]   Seed